# RetailMind AI — Day 14: LLM Review Summaries and Support Chat

This notebook uses the reusable module in `src/day14_llm_assistant.py` to summarize product reviews and run a history-aware support assistant grounded in product and policy text. Run cells from top to bottom.

**Before starting:** install `requirements.txt` and provide a review CSV. Set `OPENAI_API_KEY` in your environment before enabling the chat request. Amazon Polarity does not identify products, so product-specific summaries require a CSV with a product ID/name column.

In [ ]:
from pathlib import Path
import os
from src.day14_llm_assistant import (
    _find_reviews, load_product_reviews, summarize_reviews, _save_summaries,
    OpenAIChatClient, RetailSupportAssistant,
)

ROOT = Path.cwd()
REVIEW_CSV = _find_reviews()
print(f'Reviews: {REVIEW_CSV}')

## 1. Load reviews and inspect the product groups

The loader recognizes common review and product column names. If no product identifier is available, it places all reviews in one `all_reviews` group.

In [ ]:
product_reviews = load_product_reviews(REVIEW_CSV)
[(product, len(reviews)) for product, reviews in product_reviews.items()]

## 2. Summarize themes with Transformers

The first run downloads the summarization model. Summaries are saved as JSON for review before they are used in customer-facing content.

In [ ]:
import importlib.util
if importlib.util.find_spec("transformers") is None:
    summaries = []
    print("Review summarization skipped: Transformers is not installed; no model download attempted.")
else:
    summaries = summarize_reviews(product_reviews, model_name='sshleifer/distilbart-cnn-12-6')
    summary_path = _save_summaries(summaries)
    print(f'Saved summaries to {summary_path}')
    summaries

## 3. Prepare grounded support knowledge

Put the product facts, delivery details, and return policies the assistant may cite in `data/product_policy.txt`. The assistant is instructed to say when that information does not answer a question.

In [ ]:
knowledge_path = ROOT / 'data' / 'product_policy.txt'
if knowledge_path.is_file():
    support_knowledge = knowledge_path.read_text(encoding='utf-8')
    print(support_knowledge[:1000])
else:
    support_knowledge = (
        'Example product: Everyday Tote. Material: cotton canvas. Capacity: 18 L.\n'
        'Example policy: Unused items may be returned within 30 days with proof of purchase.'
    )
    print('Using example knowledge. Add data/product_policy.txt for your own product and policy details.')

## 4. Ask the assistant (optional API call)

This cell is opt-in because it sends a prompt to your configured LLM provider and may incur API charges. Set `RUN_CHAT = True` only when `OPENAI_API_KEY` is configured. Messages remain in this assistant instance's in-memory history.

In [ ]:
RUN_CHAT = False
if RUN_CHAT:
    if not os.getenv('OPENAI_API_KEY'):
        raise EnvironmentError('Set OPENAI_API_KEY before enabling the chat request.')
    assistant = RetailSupportAssistant(support_knowledge, OpenAIChatClient())
    print(assistant.respond('What is the tote made from?'))
    print(assistant.respond('How much can it hold?'))
else:
    print('Chat API call skipped. Set RUN_CHAT = True to send a request.')

## Saved output refresh

This cell displays the saved metrics and figures from `reports/` for this notebook. Training and generation cells are kept intact. Model-heavy workflows could not be rerun here because the required TensorFlow, PyTorch, or Transformers packages are unavailable; opt-in generation cells remain disabled.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, SVG, display

# Re-display the saved metrics and figures for this notebook's project day.
# Set ROOT to the repository root if the notebook is launched elsewhere.
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
REPORT_DIR = ROOT / "reports"
DAYS = [14]
print("Showing saved report artifacts for project day(s):", DAYS)
for day in DAYS:
    for path in sorted(REPORT_DIR.glob(f"day{day}*")):
        if path.is_dir():
            for image_path in sorted(path.rglob("*.png")):
                display(Image(filename=str(image_path)))
        elif path.suffix.lower() == ".csv":
            print("\n" + path.name)
            display(pd.read_csv(path, nrows=25))
        elif path.suffix.lower() == ".png":
            display(Image(filename=str(path)))
        elif path.suffix.lower() == ".svg":
            display(SVG(filename=str(path)))


Showing saved report artifacts for project day(s): [14]


No saved metric, image, or benchmark artifacts are available for these days. Day 14 summarization requires Transformers and a model download; the chat API request is opt-in.
